# Tarea 1: Extracción de datos fenotípicos — GSE116378

Dataset: DUTCHSCZ — Metilación del ADN en sangre de controles y pacientes con esquizofrenia.  
Fuente: NCBI GEO — https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE116378  
Plataforma: Illumina HumanMethylation450 BeadChip  
Objetivo: Obtener los metadatos clínicos de cada muestra, equivalente al phenoData de R.

In [2]:
import GEOparse
import pandas as pd
import os

# Descarga el archivo desde los servidores de NCBI GEO y lo parsea
gse = GEOparse.get_GEO(geo="GSE116378", destdir="./data_geo")

# Verificacion basica
print("Numero de muestras:", len(gse.gsms))
print("Primeras 3 muestras:", list(gse.gsms.keys())[:3])

# Extraccion de los metadatos fenotipicos de cada muestra
registros = []

for nombre_gsm, muestra in gse.gsms.items():
    fila = {"gsm_id": nombre_gsm}
    for campo, valor in muestra.metadata.items():
        fila[campo] = valor[0] if isinstance(valor, list) and len(valor) == 1 else valor
    registros.append(fila)

pheno_data = pd.DataFrame(registros)

# Seleccion de columnas relevantes
columnas_clinicas = [
    col for col in pheno_data.columns
    if "characteristics" in col or col in ("gsm_id", "title")
]

pheno_limpio = pheno_data[columnas_clinicas].copy()

# Desempaquetar characteristics_ch1 en columnas individuales
def parsear_characteristics(lista):
    resultado = {}
    if isinstance(lista, list):
        for item in lista:
            if ":" in item:
                clave, valor = item.split(":", 1)
                resultado[clave.strip()] = valor.strip()
    return resultado

caracteristicas = pheno_limpio["characteristics_ch1"].apply(parsear_characteristics)
caracteristicas_df = pd.DataFrame(caracteristicas.tolist())

# Construccion del dataframe final
pheno_final = pd.concat(
    [pheno_limpio[["gsm_id", "title"]].reset_index(drop=True),
     caracteristicas_df.reset_index(drop=True)],
    axis=1
)

# Vista completa del resultado
print("\nDimensiones del phenoData final:", pheno_final.shape)
print("\nColumnas finales:", pheno_final.columns.tolist())
print("\nPhenoData completo:")
print(pheno_final.to_string())

# Exportacion a CSV
os.makedirs("../resultados", exist_ok=True)
ruta_salida = "../resultados/GSE116378_phenodata.csv"
pheno_final.to_csv(ruta_salida, index=False)
print("\nArchivo guardado en:", ruta_salida)

10-Apr-2026 16:40:26 DEBUG utils - Directory ./data_geo already exists. Skipping.
10-Apr-2026 16:40:26 INFO GEOparse - File already exist: using local version.
10-Apr-2026 16:40:26 INFO GEOparse - Parsing ./data_geo\GSE116378_family.soft.gz: 
10-Apr-2026 16:40:26 DEBUG GEOparse - DATABASE: GeoMiame
10-Apr-2026 16:40:26 DEBUG GEOparse - SERIES: GSE116378
10-Apr-2026 16:40:26 DEBUG GEOparse - PLATFORM: GPL13534
c:\Users\LENOVO\Desktop\Cursor\Uni\ProyectoGrado\epigenetica_computacional\venv_tarea1\Lib\site-packages\GEOparse\GEOparse.py:401: DtypeWarning: Columns (0: CHR, 1: Chromosome_36, 2: Coordinate_36, 3: SPOT_ID) have mixed types. Specify dtype option on import or set low_memory=False.
  return read_csv(StringIO(data), index_col=None, sep="\t")
10-Apr-2026 16:40:38 DEBUG GEOparse - SAMPLE: GSM3230147
10-Apr-2026 16:40:38 DEBUG GEOparse - SAMPLE: GSM3230148
10-Apr-2026 16:40:38 DEBUG GEOparse - SAMPLE: GSM3230149
10-Apr-2026 16:40:38 DEBUG GEOparse - SAMPLE: GSM3230150
10-Apr-2026 16:

Numero de muestras: 64
Primeras 3 muestras: ['GSM3230147', 'GSM3230148', 'GSM3230149']

Dimensiones del phenoData final: (64, 13)

Columnas finales: ['gsm_id', 'title', 'tissue', 'group', 'gender', 'age', 'cd8t', 'cd4t', 'nk', 'bcell', 'mono', 'sentrix', 'position']

PhenoData completo:
        gsm_id                                title       tissue group gender age                cd8t                cd4t                  nk               bcell                mono     sentrix position
0   GSM3230147         DUTCHSCZ-01: Blood_Control_1  Whole blood   CTR      F  27   0.114405658860362   0.182269209534099  0.0405247085359864  0.0821755884025784   0.104378038771865  9373550151   R02C02
1   GSM3230148         DUTCHSCZ-02: Blood_Control_2  Whole blood   CTR      F  24  0.0416757400422807    0.12448009016605  0.0517446507702588  0.0349972112175631  0.0749324122133966  9373550151   R03C01
2   GSM3230149         DUTCHSCZ-03: Blood_Control_3  Whole blood   CTR      F  23   0.116986325635545  